In [2]:
import json
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# 1. Veri setindeki ham metinleri ayıklama
dataset_path = "animasyon_dataset.json"

with open(dataset_path, "r", encoding="utf-8") as f:
    data = json.load(f)

train_texts = []
for item in data:
    for msg in item["messages"]:
        if "content" in msg and msg["content"]:
            train_texts.append(msg["content"])
        if "thinking" in msg and msg["thinking"]:
            train_texts.append(msg["thinking"])

corpus_path = "verimetni.txt"
with open(corpus_path, "w", encoding="utf-8") as f:
    for text in train_texts:
        f.write(text + "\n")

print(f"Eğitim kümesi hazırlandı. Toplam satır sayısı: {len(train_texts)}")

# 2. BPE Tokenizer Yapılandırması ve Eğitimi
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))
tokenizer.pre_tokenizer = Whitespace()

special_tokens = ["[UNK]", "[PAD]", "[CLS]", "[SEP]", "[MASK]", "<|im_start|>", "<|im_end|>"]
trainer = BpeTrainer(
    vocab_size=128000,
    special_tokens=special_tokens,
    min_frequency=2
)

print("BPE Tokenizer eğitiliyor...")
tokenizer.train([corpus_path], trainer)

# 3. Doğrudan Standart Hugging Face Yapısında Kaydetme
save_directory = "./animasyon_bpe_tokenizer"
os.makedirs(save_directory, exist_ok=True)

# Hugging Face hub ve fine-tune modellerinin aradığı asıl dosya "tokenizer.json"dır.
tokenizer.save(os.path.join(save_directory, "tokenizer.json"))

print(f"[MÜKEMMEL] Tokenizer sıfır hata ile '{save_directory}/tokenizer.json' olarak kaydedildi!")

Eğitim kümesi hazırlandı. Toplam satır sayısı: 3060
BPE Tokenizer eğitiliyor...
[MÜKEMMEL] Tokenizer sıfır hata ile './animasyon_bpe_tokenizer/tokenizer.json' olarak kaydedildi!


In [3]:
import json

# Belirlediğin özel tokenlerin hepsini kapsayan konfigürasyon
config = {
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "unk_token": "[UNK]",
    "pad_token": "[PAD]",
    "cls_token": "[CLS]",
    "sep_token": "[SEP]",
    "mask_token": "[MASK]",
    "clean_up_tokenization_spaces": False,
    "tokenizer_class": "PreTrainedTokenizerFast"
}

with open("./animasyon_bpe_tokenizer/tokenizer_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

# Özel token eşleme haritası
special_map = {
    "bos_token": "<|im_start|>",
    "eos_token": "<|im_end|>",
    "unk_token": "[UNK]",
    "pad_token": "[PAD]",
    "cls_token": "[CLS]",
    "sep_token": "[SEP]",
    "mask_token": "[MASK]"
}

with open("./animasyon_bpe_tokenizer/special_tokens_map.json", "w", encoding="utf-8") as f:
    json.dump(special_map, f, ensure_ascii=False, indent=2)

print("Özel tokenlerine tam uyumlu yapılandırma dosyaları başarıyla klasöre eklendi!")

Özel tokenlerine tam uyumlu yapılandırma dosyaları başarıyla klasöre eklendi!


In [5]:
from tokenizers import Tokenizer
from huggingface_hub import hf_hub_download

# Tokenizer dosyasını Hugging Face deposundan otomatik indirme
tokenizer_path = hf_hub_download(repo_id="meldakahramann/animasyon-bpe-tokenizer", filename="tokenizer.json")

# Tokenizer'ı indirdiğimiz dosya üzerinden yükleme
tokenizer = Tokenizer.from_file(tokenizer_path)

# Örnek bir kullanıcı sorusu
ornek_soru = "Buz Devri filminin konusu nedir?"

# Metni token'larına ayırma ve encode etme
output = tokenizer.encode(ornek_soru)

# Çıktıları ekrana yazdırma
print("Tokenlar:", output.tokens)
print("Token ID'leri:", output.ids)

Tokenlar: ['Buz', 'Devri', 'filminin', 'konusu', 'nedir', '?']
Token ID'leri: [527, 633, 158, 534, 531, 28]
